In [1]:
!pip install ultralytics

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.8/45.8 kB 1.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 21.7 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 65.3/65.3 kB 2.7 MB/s eta 0:00:00


In [2]:
%%writefile process_yolo.py
import os
import json
import torch
import multiprocessing as mp
import queue
from ultralytics import YOLO

# Cấu hình danh sách thư mục input
L21 = '/kaggle/input/datasets/bumbleboo/aic26-b2-taylor/dataset/Keyframes_L21'
RUN_DIRS = [L21]
OUTPUT_DIR = "/kaggle/working/object_detection"

def process_worker(task_queue, device_id, output_dir, model_name, conf_threshold):
    """
    Hàm worker liên tục lấy việc từ task_queue ra làm cho đến khi hết việc.
    """
    print(f"[GPU {device_id}] Đã khởi động, đang load model...")
    model = YOLO(model_name)
    
    while True:
        try:
            # Lấy 1 công việc từ hàng đợi (nếu rỗng, đợi tối đa 3 giây)
            task = task_queue.get(timeout=3)
        except queue.Empty:
            break
            
        if task is None:
            break
            
        video_folder, keyframes_dir = task
        video_path = os.path.join(keyframes_dir, video_folder)
        json_filepath = os.path.join(output_dir, f"{video_folder}.json")
        
        # Nếu file json đã tồn tại, bỏ qua để tránh chạy lại khi đứt gánh
        if os.path.exists(json_filepath):
            print(f"[GPU {device_id}] Bỏ qua {video_folder} vì đã tồn tại file json.")
            continue
            
        video_results = []
        n = 1 

        for image_file in sorted(os.listdir(video_path)):
            if not image_file.lower().endswith((".jpg", ".jpeg", ".png")):
                continue

            image_path = os.path.join(video_path, image_file)
            
            # --- KIỂM TRA ẢNH LỖI (CHỐNG CRASH) ---
            if os.path.getsize(image_path) == 0:
                print(f"[GPU {device_id}] CẢNH BÁO: Ảnh 0 bytes: {image_file}, bỏ qua!")
                n += 1
                continue
                
            try:
                results = model(image_path, device=f'cuda:{device_id}', verbose=False)[0]
            except Exception as e:
                print(f"[GPU {device_id}] CẢNH BÁO: Lỗi đọc ảnh {image_file}, bỏ qua! Chi tiết: {e}")
                n += 1
                continue
            # --------------------------------------

            processed_objects = {}
            boxes = results.boxes.xyxyn.cpu().numpy()
            confs = results.boxes.conf.cpu().numpy()
            class_ids = results.boxes.cls.cpu().numpy()
            class_names = results.names

            for i in range(len(confs)):
                conf = float(confs[i])
                if conf < conf_threshold:
                    continue

                entity_name = class_names[int(class_ids[i])].capitalize()
                xmin, ymin, xmax, ymax = boxes[i]
                area = float((xmax - xmin) * (ymax - ymin))

                if entity_name not in processed_objects:
                    processed_objects[entity_name] = {"count": 0, "sum_confidence": 0.0, "total_area": 0.0}

                processed_objects[entity_name]["count"] += 1
                processed_objects[entity_name]["sum_confidence"] += conf
                processed_objects[entity_name]["total_area"] += area

            final_results = {}
            for obj, stats in processed_objects.items():
                final_results[obj] = {
                    "confidence": round(stats["sum_confidence"] / stats["count"], 4),
                    "total_area": round(stats["total_area"], 4),
                    "count": stats["count"]
                }

            frame_name = os.path.splitext(image_file)[0]
            video_results.append({
                "n": n,
                "frame": frame_name,
                "objects": final_results
            })
            n += 1

        if video_results:
            with open(json_filepath, 'w', encoding='utf-8') as f:
                json.dump(video_results, f, ensure_ascii=False, indent=4)
                
        print(f"[GPU {device_id}] Hoàn thành: {video_folder}")
        
    print(f"[GPU {device_id}] Đã hết việc, tắt worker!")


def main():
    mp.set_start_method('spawn', force=True)
    os.makedirs(OUTPUT_DIR, exist_ok=True)
    
    all_tasks = []
    for parent_dir in RUN_DIRS:
        if not os.path.exists(parent_dir):
            continue
            
        for folder_name in sorted(os.listdir(parent_dir)):
            if os.path.isdir(os.path.join(parent_dir, folder_name)):
                all_tasks.append((folder_name, parent_dir))

    num_gpus = torch.cuda.device_count()
    if num_gpus == 0:
        num_gpus = 1
    elif num_gpus > 2:
        num_gpus = 2

    print(f"Tổng số lượng thư mục video cần xử lý: {len(all_tasks)}")
    print(f"Sử dụng {num_gpus} GPU(s) với cơ chế Hàng Đợi (Queue).")

    task_queue = mp.Queue()
    for task in all_tasks:
        task_queue.put(task)
        
    for _ in range(num_gpus):
        task_queue.put(None)

    processes = []
    for i in range(num_gpus):
        p = mp.Process(
            target=process_worker, 
            args=(task_queue, i, OUTPUT_DIR, 'yolo11x.pt', 0.4)
        )
        p.start()
        processes.append(p)

    for p in processes:
        p.join()
        
    print("Hoàn tất xử lý toàn bộ các video trên tất cả GPU!")

if __name__ == '__main__':
    main()

Writing process_yolo.py


In [ ]:
!python process_yolo.py

In [ ]:
!zip -r /kaggle/working/object_detection.zip /kaggle/working/object_detection

In [ ]:
from IPython.display import FileLink

# Tạo link tải trực tiếp
FileLink(r'object_detection.zip')